# Guessing transition states with force field interpolation

Finding transition states is of crucial importance for computational chemistry, and there are many methods for doing so. However, most of these methods require some initial guess of the transition state structure. If the guess is not good enough, the method might not converge. Reaction Force Field interpolation allows for generating such guesses at a very cheap computational level by relying mostly on molecular mechanics.

Force field interpolation works by describing the reactant and product states with classical force fields $E_{1}$ and $E_{2}$ respectively. By taking a linear combination 

$$V(\lambda) = (1-\lambda)E_{1} + \lambda E_{2}$$

one can crudely describe the Potential Energy Surface (PES) along the reaction. By doing an MM optimisation of the structure at various values of $\lambda$, the relative MM energy can be obtained. The optimised structure can also be used to perform a single-point quantum mechanical (QM) calculation, which will give an more accurate energy. The structure at which these energies are highest is a good guess for the transition state structure. This workflow can be executed with just a couple lines of code. Let's consider the Bromine substitution of Ethyl chloride:

$$\mathrm{C_2H_5Cl + Br^- \rightarrow C_2H_5Br + Cl^-}$$

The following code will perform the entire workflow and visualise the results:

In [1]:
import veloxchem as vlx

In [2]:
import veloxchem as vlx

rea1 = vlx.Molecule.read_smiles("[Br]")
rea2 = vlx.Molecule.read_smiles("CCCl")
rea1.set_charge(-1)

pro1 = vlx.Molecule.read_smiles("[Cl]")
pro2 = vlx.Molecule.read_smiles("CCBr")
pro1.set_charge(-1)

tsguesser = vlx.TransitionStateGuesser()
results = tsguesser.find_transition_state([rea1, rea2], [pro1, pro2])
tsguesser.show_results(results)

* Info * Building forcefields. Disable mute_ff_build to see detailed output.                                              
                                                     Reaction summary                                                     
                                                    1 breaking bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                                          c3       c3    3  -    cl       cl    4                                         
                                                                                                                          
                                                     1 forming bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                

interactive(children=(SelectionSlider(description='Lambda', index=10, options=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25…

To better understand the workflow and it's limitations, let's go through it step by step.

## Step by step calculation of $S_N 2$ reaction



To create interpolated forcefields, we need to provide reactant and product structures as molecule objects. The program will then calculate partial charges, but it is also possible to provide those manually. The program figures out which bonds are broken and formed. It then does an MM optimisation of the combined reactand and product systems which will be used as starting points for the scan.

In [3]:
rea1 = vlx.Molecule.read_smiles("[Br]")
rea2 = vlx.Molecule.read_smiles("CCCl")
rea1.set_charge(-1)

pro1 = vlx.Molecule.read_smiles("[Cl]")
pro2 = vlx.Molecule.read_smiles("CCBr")
pro1.set_charge(-1)
tsguesser = vlx.TransitionStateGuesser()

tsguesser.build_forcefields([rea1, rea2], [pro1, pro2])

* Info * Building forcefields. Disable mute_ff_build to see detailed output.                                              
                                                     Reaction summary                                                     
                                                    1 breaking bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                                          c3       c3    3  -    cl       cl    4                                         
                                                                                                                          
                                                     1 forming bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                

{'breaking_bonds': {(2, 3)},
 'forming_bonds': {(0, 2)},
 'static_bonds': {(1, 2), (1, 4), (1, 5), (1, 6), (2, 7), (2, 8)},
 'reactant': <veloxchem.mmforcefieldgenerator.MMForceFieldGenerator at 0x7fbb2f4aec90>,
 'product': <veloxchem.mmforcefieldgenerator.MMForceFieldGenerator at 0x7fbb2f4358e0>}

The optimised structures can be inspected as follows:

In [4]:
tsguesser.reactant.molecule.show()
tsguesser.product.molecule.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

If this looks reasonable, we can proceed to scan the reaction. By default, the reaction is split up in 21 steps. On every step, the structure is minimised to the new forcefield, then sampled for 1000 steps at 600K, and then optimised again. If the value of $E1$ decreases (or $E2$ increases), this indicates that the obtained structures are could be part of different valleys in the energy landscape. The program will then try to do a conformer search to find the lowest lying conformer.

In [6]:
tsguesser.build_systems()
mm_results = tsguesser.scan_mm()

* Info * Building MM systems for the transition state guess. Disable mute_ff_build to see detailed output.                


                                                                                                                          
* Info * Saving systems as xml to ts_data_1775638402/systems                                                              
* Info * Saving systems to /home/david/KTH_Postdoc/echem/docs/mol_struct/ts_data_1775638402/systems                       
                                                                                                                          
                                                     Starting MM scan                                                     
                                            MD steps:                    1000                                             
                                            conf. snapshots:                1                                             
                                            MD temperature:             600 K                                             
                

The results can then be visualised.

In [ ]:
tsguesser.show_results(mm_results)

interactive(children=(SelectionSlider(description='Lambda', index=10, options=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25…

While the MM energies give a reasonable description of the energetic contributions from coloumbic and van der Waals interactions, they do not capture bond breaking and formation. To get a better estimate of the energy, we can perform single-point QM calculations on the MM optimised structures. Here, we use DFT with the B3LYP functional and the 'def2-SVP' basis set which are the default settings.

In [10]:
tsguesser.scf_xcfun = 'b3lyp'
tsguesser.scf_basis = 'def2-svp'
scf_results = tsguesser.scan_qm()

* Info * Disable mute_scf to see detailed output.                                                                         
                                                                                                                          
                                                     Starting QM scan                                                     
                                                                                                                          
                                                      QM parameters:                                                      
                                                 Basis:         def2-svp                                                  
                                                 DFT xc fun:        PBE0                                                  
                                                 Max conf.:            5                                                  
                

In [ ]:
tsguesser.show_results(scf_results)

interactive(children=(SelectionSlider(description='Lambda', index=11, options=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25…

And these results can be visualised as well. Note how in this (simple) example, the peak of the MM energies coincides with the SCF peak. In more complex reactions, this is not always the case, but it is usually close and the MM peak might already be a good enough guess.

Then we can proceed with a proper QM transition state optimisation.

In [12]:
basis = vlx.MolecularBasis.read(tsguesser.molecule, "def2-svp")
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.ostream.mute()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(tsguesser.molecule, basis)
scf_opt_drv = vlx.OptimizationDriver(scf_drv)
scf_opt_drv.ostream.mute()
scf_opt_drv.transition = True  #We set the option to find transition states
opt_results = scf_opt_drv.compute(tsguesser.molecule, basis, scf_results)
vlx.Molecule.read_xyz_string(opt_results['final_geometry']).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Conformer searching

Some systems might have many conformers along the PES, and in order to find the correct transition state, it is important to find the lowest lying conformer. When `peak_conformer_search` is set to `True`, a conformer search will be performed at the peak of the PES over a range of $\lambda$ values. It is also possible to automatically perform a conformer search when a discontinuity in the PES is detected by setting `discont_conformer_search` to `True`. Lastly, setting `force_conformer_search` to `True` will perform a conformer search at every $\lambda$ value. All of these functions are turned off by default because they are more time consuming. Depending on the system and the use case, the user has to decide which options are appropriate. In the following example, the longer carbon backbone gives the system more conformational freedom. Rescanning the entire PES is not necessary, but performing a conformer search at the peak is useful to obtain the correct conformer for the transition state guess.

In [13]:
rea1 = vlx.Molecule.read_smiles("[Br]")
rea2 = vlx.Molecule.read_smiles("CC(C)CC(Cl)C")
rea1.set_charge(-1)

pro1 = vlx.Molecule.read_smiles("[Cl]")
pro2 = vlx.Molecule.read_smiles("CC(C)CC(Br)C")
pro1.set_charge(-1)

tsguesser = vlx.TransitionStateGuesser()

tsguesser.build_forcefields([rea1, rea2], [pro1, pro2])
tsguesser.build_systems()

tsguesser.peak_conformer_search = True
tsguesser.peak_conformer_search_range = 2
results = tsguesser.scan_mm()
tsguesser.show_results(results)

* Info * Building forcefields. Disable mute_ff_build to see detailed output.                                              


                                                     Reaction summary                                                     
                                                    1 breaking bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                                          c3       c3    6  -    cl       cl    7                                         
                                                                                                                          
                                                     1 forming bonds:                                                     
                                       ReaType  ProType  ID - ReaType  ProType  ID                                        
                                          br       br    1  -    c3       c3    6                                         
                

interactive(children=(SelectionSlider(description='Lambda', index=10, options=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25…

## Forced bond breaking

To correctly interpolate between the reactant and the product, each atom in the reactant and product need to match. This mapping is detected automatically, but can be provided manually. From this mapping the breaking and forming bonds are derived. It is even possible to specify bonds that should be forced to break which are then not allowed to reconnect. This is useful for example in certain catalysis reactions, where both the reactant and the product contain the same catalyst molecule. The optimal mapping would not break or form any bonds with these molecules, but the reaction of interest should involve this catalyst. The following example demonstrates this with methanol catalysing the proton transfer of a larger complex. In order to run the example faster, the partial charges are pre-computed and provided manually.

In [15]:
evb = vlx.EvbDriver()

q_rea = [[
    -0.63376,
    0.155898,
    0.40852,
    0.023114,
    0.023114,
    0.023114,
],
         [
             -0.053121, -0.224161, -0.146708, -0.198581, -0.104851, 0.057048,
             0.211108, -0.00552, 0.582421, -0.578475, -0.239007, -0.200459,
             0.076475, 0.046494, -0.014331, -0.064532, -0.241586, -0.879622,
             0.102603, 0.135941, 0.12827, 0.137736, 0.085068, 0.010794,
             0.056581, 0.118934, 0.118934, 0.118934, 0.071642, 0.071642,
             0.084769, 0.084769, 0.084769, 0.068894, 0.068894, 0.068894,
             0.11978, 0.11978, 0.11978
         ]]
q_pro = [
    [
        -0.637505,
        0.152377,
        0.411487,
        0.024547,
        0.024547,
        0.024547,
    ],
    [
        -0.088958, -0.19184, -0.147759, -0.194697, -0.060166, 0.060322,
        0.057387, -0.279844, 0.484962, -0.274851, -0.152366,
        -0.7389260000000002, 0.034466, -0.007965, -0.126105, -0.288341,
        -0.009113, -0.684542, 0.112558, 0.144675, 0.133772, 0.134584, 0.1315,
        0.137143, 0.100035, 0.100035, 0.100035, 0.071572, 0.071572, 0.100344,
        0.100344, 0.100344, 0.139586, 0.139586, 0.139586, 0.066736, 0.066736,
        0.066736, 0.450857
    ],
]

rea1 = vlx.Molecule.read_smiles("OC")
rea2 = vlx.Molecule.read_smiles("C1=CC=CC=C1C(C(C(=O)OC)C[N+](C)(C)C)([O-])")

pro1 = vlx.Molecule.read_smiles("OC")
pro2 = vlx.Molecule.read_smiles("C1=CC=CC=C1C(C(=C(OC)[O-])C[N+](C)(C)C)O")

rea_charges = q_rea
pro_charges = q_pro
breaking_bonds = {(0, 2)}
tsguesser = vlx.TransitionStateGuesser()
tsguesser.build_forcefields(
    [rea1, rea2],
    [pro1, pro2],
    forced_breaking_bonds={(1, 3)},
    reactant_partial_charges=rea_charges,
    product_partial_charges=pro_charges,
)

* Info * Building forcefields. Disable mute_ff_build to see detailed output.                                              


ValueError: Could not find a mapping between the reactant and product force fields.

The output shows that the program found the correct forming and breaking bonds. We can then proceed with the mm scan.

In [16]:
tsguesser.build_systems()
mm_results = tsguesser.scan_mm()
tsguesser.show_results(mm_results)

AttributeError: 'TransitionStateGuesser' object has no attribute 'reactant'

## Available parameters and options

There are a bunch more options available to customise the TS-guesser. Below is a complete list of all options with their default values.



To be added.